In [7]:
import torch
import numpy as np
from tqdm import tqdm
import os
from torchvision import transforms
from safetensors.torch import load_file

# ===================== 强制彻底离线（无任何网络请求）=====================
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TIMM_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"

# ===================== 配置 =====================
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
IMAGE_SIZE = 224

NPZ_DIR = "/root/autodl-tmp/figure_separate"
SAVE_FEATURE_DIR = "/root/autodl-tmp/f_s_mae_features"
# 你本地已下载的MAE权重（完全正确）
LOCAL_WEIGHT = "/root/autodl-tmp/new_mae_model/model.safetensors"

os.makedirs(SAVE_FEATURE_DIR, exist_ok=True)

# ===================== 🔥 匹配你下载的模型：vit_base_patch16_224.mae =====================
print("🚀 纯离线加载 你下载的 MAE 模型...")

import timm
# ✅ 模型名必须和你下载的 repo 完全一致！
model = timm.create_model(
    "vit_base_patch16_224.mae",  # 这才是你下载的模型名！
    pretrained=False,             # 纯离线，不联网
    img_size=224,
    num_classes=0                 # 🔥 关键：输出768维特征，不是分类结果
)

# 加载你本地已经下载好的权重
state_dict = load_file(LOCAL_WEIGHT)
model.load_state_dict(state_dict, strict=False)

# ✅ 不再需要 .encoder！直接用 model 本身作为特征提取器
model = model.to(DEVICE)
model.eval()

print("✅ MAE 模型加载成功！设备：", DEVICE)
print("✅ 确认：模型是 vit_base_patch16_224.mae，直接提取特征！")

# ===================== 预处理 =====================
preprocess = transforms.Compose([
    transforms.ToTensor(),
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# ===================== MAE 特征提取（修复版）=====================
def extract_mae(img_np):
    if img_np.ndim == 2:
        img_np = np.stack([img_np]*3, axis=-1)  # 灰度图转RGB
    with torch.no_grad():
        img = preprocess(img_np).unsqueeze(0).to(DEVICE)
        feat = model(img).squeeze(0).cpu().numpy()  # 输出形状: (768,)
    return feat

# ===================== 批量处理 =====================
def extract_all():
    files = [f for f in os.listdir(NPZ_DIR) if f.endswith(".npz")]
    print(f"📊 共 {len(files)} 个股票文件")
    
    for f in tqdm(files, desc="MAE特征提取"):
        data = np.load(os.path.join(NPZ_DIR, f))
        imgs, labels = data["cv"], data["label"]
        feats = [extract_mae(img) for img in imgs]
        
        np.savez_compressed(
            os.path.join(SAVE_FEATURE_DIR, f"{f[:-4]}_mae.npz"),
            feature=np.array(feats), label=labels
        )
    print("🎉 MAE特征提取全部完成！")

if __name__ == "__main__":
    extract_all()

🚀 纯离线加载 你下载的 MAE 模型...
✅ MAE 模型加载成功！设备： cuda
✅ 确认：模型是 vit_base_patch16_224.mae，直接提取特征！
📊 共 270 个股票文件


MAE特征提取: 100%|██████████| 270/270 [19:01<00:00,  4.23s/it]

🎉 MAE特征提取全部完成！
